# 11 — Logging, Configuration, and Error Boundaries

Use the numbered concept checks where you need practice. For the engineering route, complete **Diagnose a rejected import** and the independent build.

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "intermediate", "notebooks"))
    sys.path.insert(0, os.getcwd())

In [ ]:
import logging

from lesson import Exercises, ___, run

check = Exercises('intermediate/11', {1: '29f52b19734b477d', 2: 'c174fb1a77df2749', 3: '5169449f930d5118', 4: '892d5ba000477d8a', 5: 'faa3fadb910a113a', 6: '458ee54bc4376e66', 7: '776414ad35fcce35', 8: 'dc96134717c2693f'}, practice='intermediate_03_data_quality_and_testing', independent='parse_quantity')

## 1. Loggers and levels

- Get a named logger with `logging.getLogger(__name__)`; don't use `print` for diagnostics.
- Levels from low to high: DEBUG, INFO, WARNING, ERROR, CRITICAL. A logger set to INFO drops DEBUG.
- Pass values as arguments: `logger.info("GET %s -> %s", url, status)`. They're formatted only if the message is kept.

In [ ]:
import logging
logger = logging.getLogger("demo")
logger.setLevel(logging.INFO)
records = []
class ListHandler(logging.Handler):
    def emit(self, record):
        records.append((record.levelname, record.getMessage()))
logger.addHandler(ListHandler())
logger.debug("hidden")
logger.info("GET %s -> %s", "/users", 200)
logger.warning("slow")
print(records)

**Exercise 1.** Put the levels in order, lowest first.

In [ ]:
import logging
levels = sorted(["ERROR", "DEBUG", "WARNING", "INFO"], key=lambda name: getattr(logging, name))
predicted = ___
check(1, predicted)

**Exercise 2.** A logger set to WARNING: which of `debug`, `info`, `error` messages are kept?

In [ ]:
predicted = ___   # a list of the kept level names, e.g. ["error"]
check(2, predicted)

**Exercise 3.** Format a log message the lazy way. What does `record.getMessage()` give?

In [ ]:
import logging
record = logging.LogRecord("x", logging.INFO, "", 0, "GET %s -> %s", ("/home", 200), None)
predicted = ___
check(3, predicted)

## 2. Never log secrets

- Tokens, passwords and API keys must never reach logs; logs are copied to many places.
- Log what you need to debug (method, URL, status, timing), not the credentials.
- If something might contain a secret, redact it before logging: `"***"`.

In [ ]:
def describe(url, token, status):
    return f"GET {url} -> {status}"   # token deliberately left out

print(describe("/orders", "sk-secret", 201))

**Exercise 4.** Build the log line `GET <url> -> <status>` without the token.

In [ ]:
url, token, status = "/me", "sk-123", 200
line = f"GET {url} -> {___}"
check(4, (line, token in line))

**Exercise 5.** Redact the token in a header dict before logging it.

In [ ]:
headers = {"Authorization": "Bearer sk-123", "Accept": "json"}
safe = {k: (___ if k == "Authorization" else v) for k, v in headers.items()}
check(5, safe)

## 3. Custom errors and chaining

- Make a domain error by subclassing: `class InvalidQuantity(ValueError): pass`. Callers can catch it specifically or as `ValueError`.
- Convert low-level errors at the boundary: `raise InvalidQuantity(...) from error`. The original is kept in `__cause__`.
- Validate after parsing too: a number can parse and still be invalid (negative).

In [ ]:
class InvalidQuantity(ValueError):
    pass

def parse_quantity(text):
    try:
        value = int(text)
    except ValueError as error:
        raise InvalidQuantity(f"not a whole number: {text!r}") from error
    if value < 0:
        raise InvalidQuantity("quantity cannot be negative")
    return value

try:
    parse_quantity("two")
except InvalidQuantity as error:
    print(error, "| cause:", type(error.__cause__).__name__)

**Exercise 6.** Is an `InvalidQuantity` caught by `except ValueError`?

In [ ]:
class InvalidQuantity(ValueError):
    pass

try:
    raise InvalidQuantity("bad")
except ValueError:
    caught = True
predicted = ___
check(6, predicted)

**Exercise 7.** `raise ... from error` keeps the original in `__cause__`. Predict the cause's type name.

In [ ]:
class AppError(Exception):
    pass

try:
    try:
        int("x")
    except ValueError as error:
        raise AppError("bad input") from error
except AppError as app_error:
    cause = type(app_error.__cause__).__name__
predicted = ___   # the value of cause
check(7, predicted)

**Exercise 8.** Reject negatives after parsing.

In [ ]:
def parse_quantity(text):
    value = int(text)
    if ___:
        raise ValueError("negative")
    return value

results = []
for text in ["3", "-2", "0"]:
    try:
        results.append(parse_quantity(text))
    except ValueError:
        results.append("error")
check(8, results)

### Diagnose a rejected import

Build a quantity-import service around parse_quantity. For each invalid value, record its line number and continue. Emit one summary log and keep raw values out of logs. Show a mixed batch and an empty batch. Test the original cause of a malformed value with a direct call to the parser.

Read the ticket here and identify a failure case. After the independent build, use the engineering submission cell to implement it and run your regression tests. This ticket is reviewed from the submitted code and tests.

## Challenges

Real functions, graded by tests. Complete each implementation and run its check. The independent build supplies only the interface.

### 1. `InvalidQuantity`

A domain error that is also a `ValueError`.

In [ ]:
class InvalidQuantity(ValueError):
    """Domain error raised by parse_quantity."""

In [ ]:
check('InvalidQuantity')

### Independent build — `parse_quantity`

Order Desk accepts a nonnegative integer written as text. Return the integer; reject malformed or negative quantities with InvalidQuantity. For malformed text, preserve the conversion exception as the explicit cause.

Implement the interface below, run its check, and explain one boundary case with a test of your own.

In [ ]:
def parse_quantity(text: str) -> int:
    raise NotImplementedError

In [ ]:
check('parse_quantity')

### 3. `log_request`

Log `GET <url> -> <status>` at INFO, never the token.

In [ ]:
def log_request(logger: logging.Logger, url: str, token: str, status: int) -> None:
    """Log 'GET <url> -> <status>' at INFO level. The token must never appear in logs."""
    raise NotImplementedError

In [ ]:
check('log_request')

### Engineering submission

Diagnose a rejected import. Include your code, test results and review.

In [ ]:
# Implement the ticket and run your tests here, or submit the project files.

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** Which level suits 'disk 90% full, still working'?

   - **a)** CRITICAL
   - **b)** DEBUG
   - **c)** WARNING

**2.** Why avoid calling logging.basicConfig inside library code?

   - **a)** It's slow
   - **b)** It deletes logs
   - **c)** It configures logging globally for the whole app

**3.** What should you never log?

   - **a)** Timing
   - **b)** Passwords and tokens
   - **c)** Request status

In [ ]:
from quiz import Quiz

quiz = Quiz('intermediate/11', {1: 'b064fd745701', 2: 'd335550826ac', 3: 'c044ed206c1e'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [12 — Regular Expressions and Robust Text Parsing](12_regular_expressions.ipynb)